In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_california_housing # fetch - 데이터가 양이 많아서 다운로드 받기
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from torchvision import datasets, transforms

In [25]:
housing = fetch_california_housing()
X = housing.data
y = housing.target

In [26]:
# 데이터 표준화
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [28]:
# numpy 배열 -> PyTorch 타입 배열(텐서)로 변환하자
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [29]:
# 모델한테 전달하기 위해서 데이터 셋을 만들고 DataLoader에게 전달해야 한다.
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True) #
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False) #

In [30]:
# 4. 모델정의 -> 클래스로 만든다. nn.Module 라는 클래스를 반드시 상속받아야 한다.
# 부모클래스가 하는 일이 많을 때 이런식으로 설계를 한다.
class HousingClassifier(nn.Module):
    def __init__(self): # 생성자
        super(HousingClassifier, self).__init__() # 부모 생성자를 호출한다.
                                               # 부모 생성자 호출코드는 메소드의 젤 처음에 와야한다.
                                               # super가 부모를 뜻함. 두개의 매개변수를 전달한다.
        # 입력은닉층 (iris는 4개의 특성을 갖는다.)
        self.input = nn.Linear(8, 64)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 32) 
        self.output = nn.Linear(32, 1) # 결과가 하나임

    def forward(self, x):
        x = self.input(x)
        x = self.relu(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.output(x)
        return x

In [31]:
# 5. 모델 만들고, 손실함수, 옵티마이저
model = HousingClassifier()
criterion = nn.MSELoss() # 손실함수 - 다중분류, 소프트맥스를 제공함
optimizer = optim.Adam(model.parameters(), lr=0.001) # 옵티마이저

In [32]:
# 6. 학습
def train_model(epochs=100):
    model.train()
    for epoch in range(epochs):
        # train_loader - 배치사이즈만큼
        for inputs, labels in train_loader: # 현재 배치사이즈 16개임, 16개씩 가져온다.
            optimizer.zero_grad() # 옵티마이저 초기화
            outputs = model(inputs) # 순전파, 가중치 계산중
            loss = criterion(outputs, labels) # 손실값을 계산한다.
            loss.backward() # 오차의 역전파
            optimizer.step()
        print(f"Epochs[{epoch+1}/{epochs}], Loss:{loss.item():.4f}")

    print("학습완료")

In [33]:
# 평가는 다르게
def evaluate_model():
    # 모델을 평가모드로 변경한다.
    model.eval()
    
    with torch.no_grad(): # 그라디언트 계산 비활성화
        total_loss = 0 # 훈련셋이 예측이 잘 맞는 경우 카운트하기 위한 변수
        total_samples = 0 # 전체 개수
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_samples += inputs.size(0)
            total_loss += loss.item()*inputs.size(0)
        
        # MSE와 RMSE 계산
        avg_mse = total_loss / total_samples
        rmse = np.sqrt(avg_mse)

        print(f"테스트 데이터셋 평균 MSE : {avg_mse:.4f}")
        print(f"테스트 데이터셋 RMSE : {rmse:.4f}")

if __name__ == "__main__":
    train_model(100)
    evaluate_model()

Epochs[1/100], Loss:0.2947
Epochs[2/100], Loss:0.4060
Epochs[3/100], Loss:0.5074
Epochs[4/100], Loss:0.2816
Epochs[5/100], Loss:0.1255
Epochs[6/100], Loss:0.1725
Epochs[7/100], Loss:0.2660
Epochs[8/100], Loss:0.5117
Epochs[9/100], Loss:0.6850
Epochs[10/100], Loss:0.2138
Epochs[11/100], Loss:0.3364
Epochs[12/100], Loss:0.4034
Epochs[13/100], Loss:0.2629
Epochs[14/100], Loss:0.1427
Epochs[15/100], Loss:0.1513
Epochs[16/100], Loss:0.4793
Epochs[17/100], Loss:0.2069
Epochs[18/100], Loss:0.5295
Epochs[19/100], Loss:0.3069
Epochs[20/100], Loss:0.0785
Epochs[21/100], Loss:0.2445
Epochs[22/100], Loss:0.2773
Epochs[23/100], Loss:0.2437
Epochs[24/100], Loss:0.3312
Epochs[25/100], Loss:0.3067
Epochs[26/100], Loss:0.1630
Epochs[27/100], Loss:0.3424
Epochs[28/100], Loss:0.2198
Epochs[29/100], Loss:0.2183
Epochs[30/100], Loss:0.1420
Epochs[31/100], Loss:0.7214
Epochs[32/100], Loss:0.5736
Epochs[33/100], Loss:0.0764
Epochs[34/100], Loss:0.3117
Epochs[35/100], Loss:0.3554
Epochs[36/100], Loss:0.2667
E